# Prediction

An important aspect of data science is to find out what data can tell us about the future. What do data about climate and pollution say about temperatures a few decades from now? Based on a person's internet profile, which websites are likely to interest them? How can a patient's medical history be used to judge how well he or she will respond to a treatment?

To answer such questions, data scientists have developed methods for making *predictions*. In this chapter we will study one of the most commonly used ways of predicting the value of one variable based on the value of another.

In [1]:
import polars as pl
import numpy as np
import hvplot.polars
path_data = '../../assets/data/'


Here is a historical dataset used for the prediction of the heights of adults based on the heights of their parents. We have studied this dataset in an earlier section. The table `heights` contains data on the midparent height and child's height (all in inches) for a population of 934 adult "children". Recall that the midparent height is an average of the heights of the two parents.

In [2]:
# Data on heights of parents and their adult children
original = pl.read_csv(path_data + 'family_heights.csv', infer_schema_length=10000)
heights = pl.DataFrame([
    original.get_column('midparentHeight').alias('MidParent'),
    original.get_column('childHeight').alias('Child')
    ])

In [3]:
heights

MidParent,Child
f64,f64
75.43,73.2
75.43,69.2
75.43,69.0
75.43,69.0
73.66,73.5
…,…
66.64,64.0
66.64,62.0
66.64,61.0


In [4]:
heights.hvplot.scatter(x='MidParent', y='Child')

Scatterplot with 'MidParent' on the x-axis and 'Child' on the y-axis. The data points tend to have a positive correlation, where as the x value increases, so does the y-value. The data points do not fall along a specific line, but generally all follow this trend.

A primary reason for studying the data was to be able to predict the adult height of a child born to parents who were similar to those in the dataset. We made these predictions in Section 8.1, after noticing the positive association between the two variables. 

Our approach was to base the prediction on all the points that correspond to a midparent height of around the midparent height of the new person. To do this, we wrote a function called `predict_child` which takes a midparent height as its argument and returns the average height of all the children who had midparent heights within half an inch of the argument.

In [5]:
def predict_child(mpht):
    """Return a prediction of the height of a child 
    whose parents have a midparent height of mpht.
    
    The prediction is the average height of the children 
    whose midparent height is in the range mpht plus or minus 0.5 inches.
    """
    
    close_points = heights.filter(pl.col('MidParent').is_between(mpht-0.5, mpht + 0.5, closed="left"))
    return close_points.get_column('Child').mean()                       

We applied the function to the column of `Midparent` heights, and visualized the result.

In [6]:
# Apply predict_child to all the midparent heights

heights_with_predictions = heights.with_columns(
    heights.get_column('MidParent').map_elements(predict_child, return_dtype=float).alias('Prediction')
    )

In [7]:
# Draw the original scatter plot along with the predicted values

heights_with_predictions.hvplot.scatter(x='MidParent', y=['Child', 'Prediction'], color=['#002fa7', '#706700'])

Scatterplot with 'MidParent' on the x-axis. The previous data points are shown again in dark blue and labeled 'Child.' These dark blue data points tend to have a positive correlation; as the x value increases so does the y value. There are additionally gold data points labeled 'Prediction.' The gold data points are in the center of the blue data points and also show a positive association.

The prediction at a given midparent height lies roughly at the center of the vertical strip of points at the given height. This method of prediction is called *regression.* Later in this chapter we will see whether we can avoid our arbitrary definitions of "closeness" being "within 0.5 inches". But first we will develop a measure that can be used in many settings to decide how good one variable will be as a predictor of another.